# Lab 02 : Delta LAke Time Travel

#### Create a Sample Dataset

In [0]:
# Create sample data
data = [
 (1, "Alice", 85),
 (2, "Bob", 90),
 (3, "Charlie", 78)
]
columns = ["student_id", "name", "score"]
df = spark.createDataFrame(data, columns)
df.show()

+----------+-------+-----+
|student_id|   name|score|
+----------+-------+-----+
|         1|  Alice|   85|
|         2|    Bob|   90|
|         3|Charlie|   78|
+----------+-------+-----+



In [0]:
%sql
show catalogs

catalog
fmcg
samples
system
workspace


In [0]:
%sql
show schemas in workspace

databaseName
default
information_schema


In [0]:
%sql
show volumes in workspace.default

database,volume_name


In [0]:
%sql
CREATE VOLUME workspace.default.my_volume;

In [0]:
%sql
show volumes in workspace.default

database,volume_name
default,my_volume


### Write Data to Delta Table

In [0]:
# Save as Delta table
# df.write.format("delta").mode('overwrite').save('/tmp/delta/students') --> this will give error because Databricks has moved toward disabling DBFS root in newer/more secure workspace configurations. The reason is mainly security and governance
df.write.format("delta").mode("overwrite").save('/Volumes/workspace/default/my_volume/students')

In [0]:
df = spark.read.table("students")
df.show()

+----------+-------+-----+
|student_id|   name|score|
+----------+-------+-----+
|         1|  Alice|   85|
|         2|    Bob|   90|
|         3|Charlie|   78|
+----------+-------+-----+



In [0]:
# Read back
delta_df = spark.read.format("delta").load('/Volumes/workspace/default/my_volume/students')
delta_df.show()


+----------+-------+-----+
|student_id|   name|score|
+----------+-------+-----+
|         1|  Alice|   85|
|         2|    Bob|   90|
|         3|Charlie|   78|
+----------+-------+-----+



### Update Data (Versioning)

In [0]:
# Update Data (Versioning)
from delta.tables import DeltaTable 
# load Delta table
delta_table=DeltaTable.forPath(spark,"/Volumes/workspace/default/my_volume/students")

# update score for Alice
delta_table.update(
    condition="name='Alice'",
    set={"score":"95"}
)

# Show Updated Data
delta_table.toDF().show() 


+----------+-------+-----+
|student_id|   name|score|
+----------+-------+-----+
|         2|    Bob|   90|
|         3|Charlie|   78|
|         1|  Alice|   95|
+----------+-------+-----+



Observation : when score of Alice was updated it showed at the end of table meaning that's the updated record in latest.

### Add New Record

In [0]:
# Add New Record
new_data=[(4,"David",88)]
new_df=spark.createDataFrame(new_data,columns)

new_df.write.format("delta").mode("append").save('/Volumes/workspace/default/my_volume/students')

In [0]:
# show current table
spark.read.format("delta").load('/Volumes/workspace/default/my_volume/students').show()

+----------+-------+-----+
|student_id|   name|score|
+----------+-------+-----+
|         1|  Alice|   95|
|         2|    Bob|   90|
|         3|Charlie|   78|
|         4|  David|   88|
+----------+-------+-----+



# Query older versions

In [0]:
# check history
delta_table.history().show()

+-------+-------------------+--------------+--------------------+---------+--------------------+----+------------------+-----------------------+--------------------+-----------+-----------------+-------------+--------------------+------------+--------------------+
|version|          timestamp|        userId|            userName|operation| operationParameters| job|          notebook|queryHistoryStatementId|           clusterId|readVersion|   isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+-------------------+--------------+--------------------+---------+--------------------+----+------------------+-----------------------+--------------------+-----------+-----------------+-------------+--------------------+------------+--------------------+
|      3|2026-10-06 17:11:06|74162278981546|shashikalagupta.r...|    WRITE|{mode -> Append, ...|NULL|{3916570572513088}|   da488d53-5a76-45d...|1006-152440-qcufp...|          2|WriteSerializable|         t

In [0]:
# Read Version 0 (original data)
spark.read.format("delta").option("versionAsOf",0).load('/Volumes/workspace/default/my_volume/students').show()

+----------+-------+-----+
|student_id|   name|score|
+----------+-------+-----+
|         1|  Alice|   85|
|         2|    Bob|   90|
|         3|Charlie|   78|
+----------+-------+-----+



In [0]:
# Read Version 1 (after alice update)
spark.read.format("delta").option("versionAsOf",1).load('/Volumes/workspace/default/my_volume/students').show()

+----------+-------+-----+
|student_id|   name|score|
+----------+-------+-----+
|         2|    Bob|   90|
|         3|Charlie|   78|
|         1|  Alice|   95|
+----------+-------+-----+

